# Hybrid — SASRec + audio-content two-tower on Yambda-50M

Effective item embedding `v_j = e_j + alpha * mask_j * ContentMLP(audio_j)` fused end-to-end. `alpha` is learned: alpha->0 means pure SASRec, so its final value shows how much the audio content helps. User side is the SASRec sequence rep.

**Bars (our harness):** ItemKNN-bm25 0.0709, **SASRec 0.0735**. Goal: beat SASRec.

**Self-contained:** it downloads + filters the audio embeddings inline (~1-2 min on Kaggle), so no extra dataset is needed. Accelerator → **GPU**, Internet On.

In [ ]:
!pip install -q --no-cache-dir --upgrade "git+https://github.com/ak1232320/nndl_capstone_2026.git"

In [ ]:
# Set the data dir to the 20 GB working disk BEFORE importing ymrec (config reads it at import).
import os
os.environ["YMREC_DATA"] = "/kaggle/working/data"
import torch
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
import time, numpy as np
from ymrec.data.sequences import build_sequences
from ymrec.data.embeddings import load_aligned_embeddings

data = build_sequences(size="50m", maxlen=200)
print(f"sequences | items={data.n_items:,} eval_users={len(data.eval_pos):,}")

# Audio embeddings aligned to the same vocab (downloads 13.8 GB on first call, ~1-2 min).
t0 = time.time()
content_emb, content_mask, dim = load_aligned_embeddings(data.item_ids)
print(f"content | {content_emb.shape} dim={dim} coverage={content_mask.mean():.3f} "
      f"({time.time()-t0:.0f}s)")

In [ ]:
from ymrec.models.hybrid import train_and_eval

t0 = time.time()
model, best = train_and_eval(
    data, content_emb, content_mask,
    d=64, n_blocks=2, n_heads=1, dropout=0.2,
    epochs=120, batch_size=128, lr=1e-3, eval_every=10,
)
print(f"\ntrained in {(time.time()-t0)/60:.1f} min | learned alpha={float(model.alpha):.3f} | best: {best}")

In [ ]:
import pandas as pd
ref = {"MostPop": 0.0171, "ItemKNN-bm25": 0.0709, "SASRec": 0.0735,
       "Hybrid (ours)": round(best["ndcg@10"], 4)}
pd.Series(ref, name="NDCG@10").sort_values(ascending=False).to_frame()